# Day 7 — Error analysis and Gradio demo

This notebook grows after every Day 7 PLAN item. The current D7-01 checkpoint only prepares the unchanged shared outer holdout and obtains ordered predictions from the saved fine-tuned classifier. Reusable logic stays in `src/transformers_learning/`.

## D7-01 — Data, tensor, and leakage boundaries

The source SST-2 rows are validated and passed through the same stratified outer split used in Days 4–6. Only the outer holdout enters this notebook checkpoint; no row is used for training, validation, tuning, or checkpoint selection. Tokenization produces `input_ids` and `attention_mask` shaped `[batch, sequence]`. The saved classifier returns logits `[batch, 2]`, which become probabilities for the fixed binary labels `0 = negative` and `1 = positive`. `model.eval()` selects evaluation behavior, while `torch.no_grad()` independently prevents gradient-graph construction; neither operation trains the model.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from transformers_learning import (
    SST2_LABEL_MAP,
    adapt_sst2_split,
    build_day7_error_tables,
    build_day7_length_summary_table,
    build_day7_prediction_preview,
    ensure_sst2_train_data,
    run_day7_holdout_inference,
    save_day7_error_report,
    summarize_day7_errors,
)


In [2]:
RUN_DAY7_D7_01_INTEGRATION = True
fine_tuned_model_directory = project_root / "fine_tuned_model"
sst2_train_path = project_root / "data" / "SST-2" / "train.tsv"

print(f"Binary label mapping: {dict(SST2_LABEL_MAP)}")
print(f"Fine-tuned artifact: {fine_tuned_model_directory}")
print(f"SST-2 training source: {sst2_train_path}")


Binary label mapping: {0: 'negative', 1: 'positive'}
Fine-tuned artifact: /home/makarlistkov/projects/transformers/fine_tuned_model
SST-2 training source: /home/makarlistkov/projects/transformers/data/SST-2/train.tsv


## Explicit integration checkpoint

The flag is `False` by default because the real run needs local SST-2 data and the ignored `fine_tuned_model/` artifact, and it predicts the complete outer holdout. Set it to `True` only when those inputs are available. The small preview displays source positions so alignment does not depend on potentially duplicated dataframe index labels.

In [3]:
if not RUN_DAY7_D7_01_INTEGRATION:
    print("D7-01 integration is disabled; no model or dataset was loaded.")
else:
    local_sst2_path = ensure_sst2_train_data(sst2_train_path)
    source_dataframe = pd.read_csv(local_sst2_path, sep="\t")
    sentiment_dataframe = adapt_sst2_split(source_dataframe)
    day7_result = run_day7_holdout_inference(
        sentiment_dataframe,
        fine_tuned_model_directory=fine_tuned_model_directory,
        batch_size=16,
        progress_callback=lambda completed, total: print(
            f"Fine-tuned holdout: {completed:,}/{total:,} "
            f"({completed / total:.1%})"
        )
        if completed == total or completed % 1024 == 0
        else None,
    )
    print(f"Outer holdout rows: {len(day7_result.holdout.texts):,}")
    print(
        "Artifact used: "
        f"{day7_result.fine_tuned_model_directory}"
    )
    display(build_day7_prediction_preview(day7_result, limit=5))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Fine-tuned holdout: 1,024/13,470 (7.6%)
Fine-tuned holdout: 2,048/13,470 (15.2%)
Fine-tuned holdout: 3,072/13,470 (22.8%)
Fine-tuned holdout: 4,096/13,470 (30.4%)
Fine-tuned holdout: 5,120/13,470 (38.0%)
Fine-tuned holdout: 6,144/13,470 (45.6%)
Fine-tuned holdout: 7,168/13,470 (53.2%)
Fine-tuned holdout: 8,192/13,470 (60.8%)
Fine-tuned holdout: 9,216/13,470 (68.4%)
Fine-tuned holdout: 10,240/13,470 (76.0%)
Fine-tuned holdout: 11,264/13,470 (83.6%)
Fine-tuned holdout: 12,288/13,470 (91.2%)
Fine-tuned holdout: 13,312/13,470 (98.8%)
Fine-tuned holdout: 13,470/13,470 (100.0%)
Outer holdout rows: 13,470
Artifact used: /home/makarlistkov/projects/transformers/fine_tuned_model


,source_position,text,true_label,predicted_label
0,58587,cinematography to the outstanding soundtrack,1,1
1,60608,robust,1,1
2,48779,duly impressive in imax dimensions,1,1
3,33879,"she 's a pretty woman ,",1,1
4,19553,even the corniest and,0,0


## D7-02 — Aligned error table and FP/FN

`build_day7_error_tables` verifies one prediction per ordered holdout text, binary labels, and finite `[p(negative), p(positive)]` vectors that sum to one and agree with `argmax`. `predicted_confidence` is the probability at the predicted label; it is not calibrated certainty or a causal explanation. A **false positive** is `true=0, predicted=1`; a **false negative** is `true=1, predicted=0`. Empty FP or FN groups are valid.

In [5]:
if not RUN_DAY7_D7_01_INTEGRATION:
    print("Run the D7-01 integration cell before the D7-02 checkpoint.")
else:
    day7_tables = build_day7_error_tables(day7_result)
    print(f"All holdout rows: {len(day7_tables.all_rows):,}")
    print(f"Errors: {len(day7_tables.errors):,}")
    print(f"False positives: {len(day7_tables.false_positives):,}")
    print(f"False negatives: {len(day7_tables.false_negatives):,}")
    print("\nDeterministic first false positives:")
    display(day7_tables.false_positives.head(20))
    print("Deterministic first false negatives:")
    display(day7_tables.false_negatives.head(20))


All holdout rows: 13,470
Errors: 768
False positives: 298
False negatives: 470

Deterministic first false positives:


,source_position,text,true_label,predicted_label,probabilities,predicted_confidence,text_length
0,9920,realistically nuanced a robert de niro perform...,0,1,"(0.006142063997685909, 0.9938579201698303)",0.993858,230
1,60196,is n't much fun without the highs and lows,0,1,"(0.184543639421463, 0.8154563903808594)",0.815456,43
2,44057,"dry of humor , verve and fun",0,1,"(0.007826641201972961, 0.9921733140945435)",0.992173,29
3,25559,"short on tension , eloquence , spiritual chall...",0,1,"(0.1427142471075058, 0.857285737991333)",0.857286,142
4,16718,pretty unbelievable,0,1,"(0.06423983722925186, 0.9357601404190063)",0.935760,20
5,45091,break up,0,1,"(0.4785066545009613, 0.5214933156967163)",0.521493,9
6,65217,"passion , grief and fear",0,1,"(0.0009778551757335663, 0.9990221261978149)",0.999022,25
7,48332,for the playlist,0,1,"(0.19474966824054718, 0.805250346660614)",0.805250,17
8,59452,the dry humor,0,1,"(0.060656141489744186, 0.9393438100814819)",0.939344,14
9,23906,to lead a group of talented friends astray,0,1,"(0.011474241502583027, 0.9885257482528687)",0.988526,43


Deterministic first false negatives:


,source_position,text,true_label,predicted_label,probabilities,predicted_confidence,text_length
0,9979,a sudden lunch rush,1,0,"(0.5868480801582336, 0.41315191984176636)",0.586848,20
1,14248,that takes a stand in favor of tradition and w...,1,0,"(0.6374880075454712, 0.3625120222568512)",0.637488,52
2,66691,written for no one,1,0,"(0.9835759997367859, 0.01642395555973053)",0.983576,19
3,35599,opportunists,1,0,"(0.5050163269042969, 0.49498364329338074)",0.505016,13
4,50155,a thirteen-year-old 's book report,1,0,"(0.9734101295471191, 0.026589831337332726)",0.973410,35
5,44030,the pitfalls,1,0,"(0.9989811778068542, 0.0010188242886215448)",0.998981,13
6,32871,about kicking undead ***,1,0,"(0.8418352603912354, 0.15816472470760345)",0.841835,25
7,59390,"pat storylines , precious circumstances",1,0,"(0.5035898089408875, 0.49641019105911255)",0.503590,40
8,9902,no fantasy story,1,0,"(0.9682004451751709, 0.03179948776960373)",0.968200,17
9,14679,us to remember that life 's ultimately a gambl...,1,0,"(0.6961071491241455, 0.30389291048049927)",0.696107,83


## D7-03 — Measured summary and report

This checkpoint measures total errors, error rate, FP/FN counts, and character lengths for correct versus incorrect rows. It writes the ignored `error_analysis.txt` with deterministic examples ordered by `source_position`. The final **qualitative hypotheses** section is deliberately separate: ambiguity, negation, contrast, truncation, and annotation noise are ideas to inspect, not causal conclusions. Do not tune or select the model from outer-holdout observations.

In [ ]:
qualitative_observations = ()
error_analysis_path = project_root / "error_analysis.txt"

if not RUN_DAY7_D7_01_INTEGRATION:
    print("Run D7-01 and D7-02 before generating the D7-03 report.")
else:
    day7_summary = summarize_day7_errors(day7_tables)
    print(f"Error rate: {day7_summary.error_rate:.2%}")
    print(
        f"Errors: {day7_summary.error_count:,} "
        f"(FP={day7_summary.false_positive_count:,}, "
        f"FN={day7_summary.false_negative_count:,})"
    )
    display(build_day7_length_summary_table(day7_summary))
    saved_error_analysis_path = save_day7_error_report(
        day7_tables,
        error_analysis_path,
        qualitative_observations=qualitative_observations,
        examples_per_type=5,
    )
    print(f"Saved ignored report: {saved_error_analysis_path}")
    print(saved_error_analysis_path.read_text(encoding="utf-8"))


### Add observations only after inspection

After reviewing FP/FN examples, replace the empty `qualitative_observations` tuple with concise hypotheses such as patterns worth checking. Keep wording cautious: confidence and text length do not prove why a prediction occurred. Rerun only the D7-03 cell to regenerate the report; never use these holdout observations to tune the model.

## Next checkpoint

D7-04 will add application-neutral one-text inference formatting. D7-03 intentionally stops at descriptive error analysis and report generation.